# Notebook 1: Almgren–Chriss Model — Derivation & Efficient Frontier

This notebook derives the AC2001 optimal execution framework from first principles and visualises the mean–variance efficient frontier.

**Reference**: Almgren, R. & Chriss, N. (2001). *Optimal execution of portfolio transactions.* Journal of Risk, 3(2), 5–39.

## 1. Problem Setup

We wish to liquidate $X$ shares over a horizon $T$, split into $N$ equal intervals of length $\tau = T/N$.

Define the **holdings trajectory**:
$$x_0 = X, \quad x_N = 0, \quad n_k = x_{k-1} - x_k \;\text{(shares sold at step } k\text{)}$$

### Price dynamics
$$S_k = S_{k-1} - g\!\left(\frac{n_k}{\tau}\right)\tau - \sigma\sqrt{\tau}\,\xi_k$$

where $g(v) = \gamma v$ is the **permanent** impact and $h(v) = \eta v$ is the **temporary** impact.

### Cost decomposition
$$E[C] = \tfrac{1}{2}\gamma X^2 + \eta \sum_{k=1}^N \frac{n_k^2}{\tau}, \qquad \operatorname{Var}[C] = \sigma^2 \tau \sum_{k=1}^N x_k^2$$

The **mean–variance utility** to minimise is:
$$U(\lambda) = E[C] + \lambda\,\operatorname{Var}[C]$$

### Optimal trajectory
The Euler–Lagrange conditions yield a closed-form solution:
$$x^*(t) = X \cdot \frac{\sinh(\kappa(T-t))}{\sinh(\kappa T)}, \qquad \kappa^2 = \frac{\lambda\sigma^2}{\eta}$$

As $\lambda \to 0$: $\kappa \to 0$ and $x^* \to X(1 - t/T)$ (uniform TWAP).  
As $\lambda \to \infty$: $\kappa \to \infty$ and execution front-loads toward $t=0$.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.almgren_chriss import (
    optimal_trajectory, efficient_frontier, twap_trajectory, cost_of_trajectory
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 2. Model Parameters

Using representative parameters for a liquid large-cap US equity (cf. Almgren et al. 2005).

In [ ]:
# Position and horizon
X     = 1_000_000   # shares to liquidate
T     = 1.0         # 1 trading day
N     = 390         # 1-minute intervals

# Market parameters
sigma = 0.02        # 2% daily fractional volatility
S0    = 100.0       # arrival price ($/share)

# Impact parameters (Almgren et al. 2005 order-of-magnitude)
gamma = 2.5e-7      # permanent impact ($/share per share/day)
eta   = 2.5e-6      # temporary impact ($/share per share/day)

print(f"Position notional: ${X * S0:,.0f}")
print(f"Daily vol ($):     ${sigma * S0:.2f} per share")
print(f"τ = T/N:           {T/N*390:.1f} minute(s) per interval")

## 3. Optimal Trajectory for Different Risk Aversions

In [ ]:
lambdas_plot = {
    r'$\lambda = 10^{-9}$ (near risk-neutral)': 1e-9,
    r'$\lambda = 10^{-6}$ (moderate)':          1e-6,
    r'$\lambda = 10^{-4}$ (risk-averse)':       1e-4,
}

fig, ax = plt.subplots(figsize=(9, 4.5))

for label, lam in lambdas_plot.items():
    t, h = optimal_trajectory(X, T, N, sigma, gamma, eta, lam)
    ax.plot(t, h / 1e6, label=label, linewidth=1.8)

# TWAP benchmark
h_twap = twap_trajectory(X, N, T)
ax.plot(np.linspace(0, T, N+1), h_twap / 1e6,
        'k--', linewidth=1.5, label='TWAP (uniform)')

ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Remaining position (M shares)', fontsize=12)
ax.set_title('AC2001 Optimal Holdings Trajectories', fontsize=13)
ax.legend(fontsize=9)
ax.set_xlim(0, T)
ax.set_ylim(0)
plt.tight_layout()
plt.savefig('../data/fig_trajectories.png', dpi=150)
plt.show()

## 4. Efficient Frontier

Each $\lambda$ traces a point on the mean–variance frontier.  Points to the left-and-down are preferred.

In [ ]:
lambdas_grid = np.logspace(-10, -3, 200)
fr = efficient_frontier(X, T, N, sigma, gamma, eta, lambdas=lambdas_grid)

# TWAP cost
h_twap = twap_trajectory(X, N, T)
twap_costs = cost_of_trajectory(h_twap, T, sigma, gamma, eta, X)

fig, ax = plt.subplots(figsize=(8, 5))

# Frontier curve
ax.plot(
    fr['variances'] / 1e12,
    fr['expected_costs'] / 1e6,
    lw=2.0, color='steelblue', label='Efficient frontier'
)

# Mark λ reference points
for lam, marker in [(1e-9, 'o'), (1e-6, 's'), (1e-4, '^')]:
    t_, h_ = optimal_trajectory(X, T, N, sigma, gamma, eta, lam)
    c_ = cost_of_trajectory(h_, T, sigma, gamma, eta, X)
    ax.scatter(
        c_['variance'] / 1e12, c_['expected_cost'] / 1e6,
        s=60, zorder=5, label=f'λ = {lam:.0e}', marker=marker
    )

# TWAP point
ax.scatter(
    twap_costs['variance'] / 1e12, twap_costs['expected_cost'] / 1e6,
    s=80, color='tomato', zorder=6, marker='D', label='TWAP'
)

ax.set_xlabel(r'$\operatorname{Var}[C]$ ($\times 10^{12}$)', fontsize=12)
ax.set_ylabel(r'$E[C]$ (\$M)', fontsize=12)
ax.set_title('AC2001 Mean–Variance Efficient Frontier', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../data/fig_frontier.png', dpi=150)
plt.show()

print(f"TWAP  — E[C]: ${twap_costs['expected_cost']:,.0f}  |  Var[C]: {twap_costs['variance']:.3e}")
print(f"Frontier range:")
print(f"  min E[C]: ${fr['expected_costs'].min():,.0f}  (λ→0)")
print(f"  max E[C]: ${fr['expected_costs'].max():,.0f}  (λ→∞)")

## 5. The κ Parameter and Frontier Shape

$\kappa = \sqrt{\lambda\sigma^2/\eta}$ controls the curvature of the trajectory.  
Small $\kappa T$ → nearly linear (TWAP-like).  Large $\kappa T$ → hyperbolic front-loading.

In [ ]:
kappas = np.sqrt(lambdas_grid * sigma**2 / eta)
kappa_T = kappas * T

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogx(lambdas_grid, kappa_T, color='darkorange')
axes[0].set_xlabel(r'Risk aversion $\lambda$', fontsize=11)
axes[0].set_ylabel(r'$\kappa T$', fontsize=11)
axes[0].set_title(r'Decay parameter $\kappa T$ vs $\lambda$', fontsize=12)
axes[0].axhline(1.0, ls='--', color='grey', alpha=0.7, label=r'$\kappa T = 1$ (transition)')
axes[0].legend(fontsize=9)

axes[1].plot(fr['std_costs'] / 1e6, fr['expected_costs'] / 1e6,
             lw=2, color='steelblue')
axes[1].scatter(
    [twap_costs['std_cost'] / 1e6], [twap_costs['expected_cost'] / 1e6],
    color='tomato', s=80, zorder=5, label='TWAP'
)
axes[1].set_xlabel(r'$\sigma[C]$ = Std of cost ($\$M$)', fontsize=11)
axes[1].set_ylabel(r'$E[C]$ ($\$M$)', fontsize=11)
axes[1].set_title(r'Frontier: $E[C]$ vs $\sigma[C]$', fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6. Summary

| Quantity | Formula | Interpretation |
|---|---|---|
| $\kappa$ | $\sqrt{\lambda\sigma^2/\eta}$ | Decay rate; higher = more front-loading |
| $E[C]$ | $\frac{1}{2}\gamma X^2 + \eta\sum n_k^2/\tau$ | Average dollar cost of execution |
| $\operatorname{Var}[C]$ | $\sigma^2\tau\sum x_k^2$ | Timing risk (exposure × volatility) |
| $U(\lambda)$ | $E[C] + \lambda\operatorname{Var}[C]$ | Minimised objective |

**Key takeaway**: The efficient frontier reveals the irreducible trade-off between expected cost and timing risk.  TWAP lies *on* the frontier at $\lambda=0$.  Any $\lambda>0$ strictly improves the risk-adjusted objective by front-loading trades.